# Lesson 20: Adapt and evaluate SmolVLA

This lesson uses small code cells to show the real SmolVLA path. Run cells in order. This notebook contains complete exercise answers.

**Result:** a saved and reloaded adaptation with held-out loss and LIBERO task evaluation.

**Notebook pair:** [Learner notebook](20-smolvla-adaptation.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
import hashlib
import json
import os
import random
import time
from collections.abc import Callable, Mapping

import numpy as np
import torch
import yaml
from lerobot.policies.smolvla import SmolVLAPolicy
from lerobot.processor import load_pretrained_policy_processors
from torch.utils.data import default_collate
from transformers import AutoTokenizer

from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.interfaces import (
    ActionProcessor,
    ObservationProcessor,
    ProcessorObservation,
    VLAInputs,
)
from vla_lab.teaching.smolvla import (
    CallbackSmolVLAPolicy,
    SmolVLAAdapter,
    flow_training_pair,
    load_prepared_smolvla,
    masked_velocity_loss,
    prepare_vla_inputs,
    select_trainable_parameters,
    train_mode_with_frozen_vlm,
)
from vla_lab.teaching.training import optimizer_update

## Model map: adaptation needs a fair comparison

A pretrained VLA is a policy: it maps camera frames, state, and instruction to actions. **Adaptation** changes selected trainable weights with new training examples. This lesson uses the real published VLM and expert as the starting checkpoint. It freezes the VLM and trains the action expert, saves the result and both processor pipelines, then reloads them.

A held-out flow loss compares predictions with recorded action velocity targets on validation episodes that are separate from training episodes. Use the same examples, noise, time, and padding mask for the before and after measurements. This isolates the effect of changed weights. A lower held-out loss is evidence of better imitation on those held-out windows; it is not a guarantee of task success.

LIBERO evaluation runs the policy in a simulator. Its controller acts at 20 Hz: one relative action per control step. The dataset stores labels at 10 FPS. Those rates have different roles. Do not repeat or resample labels to make them match. Report the actual simulator episode count, successes, and failures. A policy rollout and a held-out loss answer different questions.

```text
published checkpoint ── fixed validation loss ──┐
         │                                       ├─ compare on same held-out data
         └─ expert-only updates ─ saved bundle ─ reload ─ fixed validation loss
                                                  └─ LIBERO at 20 Hz: successes / episodes
```

The full run is limited to a 48 GB GPU memory budget and two hours. Select a smaller batch or stop the run if the live resource use reaches either limit. Do not describe five updates on two examples as learning. The adaptation experiment uses a real split, a useful update budget, a held-out measurement, and a simulator score. Local assets must already exist; no download starts here.

## Exercise 20.1 — Adapt the real policy

Implement `course_input_fn(adapter, batch) -> VLAInputs`: read camera tensors in $[0,1]$ and the normalized state from the policy batch, tokenize the task with the local tokenizer, and call `prepare_vla_inputs` with the policy input config. If actions are present, add the policy-normalized action tensor.
* `batch` - the preprocessed policy batch, a `dict` whose values are tensors, except `task`: one prompt string or a list of prompt strings. It has one `(B, 3, h, w)` camera tensor for each key of `adapter.config.image_features`, `observation.state`, and, in training, `action`.
* `prepare_vla_inputs(observation, prompts, tokenizer, config)` - supplied. `observation` maps each camera key and `observation.state` to its tensor.
* Return value - the `VLAInputs` dict from Lesson 17 (`images`, `img_masks`, `lang_tokens`, `lang_masks`, `state`, plus `actions` when the batch has `action`). The hook must process images and text; do not pass an already prepared image through an identity transform.

Use the learner loss from [Lesson 17](../17-smolvla-training/17-smolvla-training.ipynb) and sampler from [Lesson 19](../19-smolvla-flow/19-smolvla-flow.ipynb). Its input hook follows [Lesson 16](../16-smolvla-execution/16-smolvla-execution.ipynb): it reads the two raw camera tensors in `[0,1]`, the policy-normalized state, and the task prompt, then calls `prepare_vla_inputs` to resize images, tokenize the prompt, and build masks. It adds normalized actions through the policy action processor for training. The experiment starts from the published checkpoint. Load the local training and validation episodes. Freeze the VLM, train the action expert, and use a real training budget. Keep a fixed held-out batch and fixed noise/time for both before and after loss. Save the policy and both processors to a new local output directory. Reload that directory and confirm that the reloaded policy gives the same fixed-batch output.

Then run LIBERO evaluation with the saved policy. The evaluation must use one relative action per 20 Hz control step. Read frames and state through the existing evaluation runner. The 10 FPS label rate describes the dataset; it is not the simulator control rate. Report the success rate over attempted episodes, keeping each failure in the denominator:

$$\text{success rate}=\frac{N_{\text{success}}}{N_{\text{attempted}}}$$

Where:
* $N_{\text{success}}$ - episodes that reached the task goal.
* $N_{\text{attempted}}$ - every started episode, including failures, timeouts, and policy errors.

**Resource cap:** at most 48 GB GPU memory and two hours. Stop at the first limit. A useful training run uses many windows across the training split and enough optimizer updates to test a change. Do not call a tiny mechanics check “learning.”

More behaviour checks for this exercise: `check_course_input_fn` and `check_learner_flow_loss` and `check_euler_to_action` in [lesson20_checks.py](lesson20_checks.py).

In [ ]:
def course_input_fn(adapter: SmolVLAAdapter, batch: ProcessorObservation) -> VLAInputs:
    tensor_batch: dict[str, torch.Tensor] = {
        key: value for key, value in batch.items() if isinstance(value, torch.Tensor)
    }
    observation: dict[str, torch.Tensor] = {}
    for key in adapter.config.image_features:
        image = tensor_batch.get(key)
        if image is None:
            raise TypeError(f"Missing camera tensor: {key}")
        observation[key] = image
    observation["observation.state"] = adapter.policy.prepare_state(tensor_batch)
    prompt_value = batch.get("task")
    if isinstance(prompt_value, str):
        prompts: str | list[str] = prompt_value
    elif isinstance(prompt_value, list) and all(isinstance(value, str) for value in prompt_value):
        prompts = prompt_value
    else:
        raise TypeError("task must be one prompt or a list of prompts")
    inputs = prepare_vla_inputs(observation, prompts, TOKENIZER, adapter.config)
    if "action" in tensor_batch:
        inputs["actions"] = adapter.policy.prepare_action(tensor_batch)
    return inputs


def learner_flow_loss(
    adapter: SmolVLAAdapter,
    batch: Mapping[str, torch.Tensor],
    noise: torch.Tensor | None = None,
    time: torch.Tensor | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    inputs = adapter.prepare_inputs(batch)
    actions = inputs["actions"]
    noise = torch.randn_like(actions) if noise is None else noise
    time = adapter.sample_time(actions.shape[0], actions.device) if time is None else time
    x_t, target = flow_training_pair(actions, noise, time)
    velocity = adapter.velocity(inputs, x_t, time)
    loss = masked_velocity_loss(velocity, target, batch["action_is_pad"], adapter.action_dim)
    return loss, {"flow_mse": loss}


def euler_to_action(
    velocity_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    noise: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    if num_steps < 1:
        raise ValueError("num_steps must be at least one")
    dt = -1.0 / num_steps
    state = noise
    for step in range(num_steps):
        time = torch.full(
            (state.shape[0],), 1.0 + step * dt, device=state.device, dtype=torch.float32
        )
        state = state + dt * velocity_fn(state, time)
    return state


assert torch.allclose(
    euler_to_action(lambda x, t: torch.full_like(x, -3.0), torch.tensor([[[-1.0]]]), 3),
    torch.tensor([[[2.0]]]),
)
print("Lesson 20 uses the learner input, loss, and sampler hooks.")

The adaptation cells are tagged in dependency order: fixed held-out baseline, expert-only training and saved bundle, then reload and LIBERO evaluation. Run the full `adaptation` experiment only when the local data, checkpoints, simulator assets, and GPU budget are ready.

In [ ]:
# A deterministic, untrained scalar field checks the sampler shape only.
noise = torch.randn(2, 10, 7)
result = euler_to_action(lambda x, t: -x, noise, 4)
assert result.shape == noise.shape and torch.isfinite(result).all()
print("sampler mechanics passed; no learning is claimed")

# More behaviour checks: lesson20_checks.py, next to this notebook.
from lesson20_checks import check_course_input_fn, check_euler_to_action, check_learner_flow_loss

check_course_input_fn(course_input_fn)
check_learner_flow_loss(learner_flow_loss)
check_euler_to_action(euler_to_action)

**Explain (2–4 sentences).** Compare held-out loss and simulator success. What result did the saved and reloaded checkpoint produce? State the limits of the task, split, seeds, and runtime budget.

In [ ]:
# Baseline: fixed validation windows and three development-state rollouts.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 2006
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla(
    "published", DATA, device=DEVICE, n_action_steps=1
)
TOKENIZER = AutoTokenizer.from_pretrained(str(DATA / "models/smolvlm2-500m-video-instruct"))
FRAME_INTERVAL_SECONDS = 0.1  # Stored action labels are 10 FPS.
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
VALID_DS = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(POLICY.config.chunk_size)))},
)
VALID_IDS = torch.linspace(0, len(VALID_DS) - 1, 8).round().long().tolist()
VALID_BATCH = default_collate([VALID_DS[index] for index in VALID_IDS])
VALID_BATCH["task"] = ["put the bowl on the plate"] * len(VALID_IDS)
VALID_BATCH = PREPROCESSOR(VALID_BATCH)
VALID_BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in VALID_BATCH.items()
}
ADAPTER = SmolVLAAdapter(POLICY, input_fn=course_input_fn)
FIXED_NOISE = torch.randn_like(ADAPTER.prepare_inputs(VALID_BATCH)["actions"])
FIXED_TIME = torch.linspace(0.1, 0.9, len(VALID_IDS), device=DEVICE)
POLICY.eval()
with torch.no_grad():
    BEFORE_LOSS, _ = learner_flow_loss(ADAPTER, VALID_BATCH, FIXED_NOISE, FIXED_TIME)
SPLITS = yaml.safe_load((CONFIGS / "eval" / "splits.yaml").read_text())
TASKS = yaml.safe_load((CONFIGS / "data" / "episodes.yaml").read_text())
STATE_IDS = SPLITS["tasks"]["bowl_plate"]["development_state_ids"][:3]
TASK_PROMPT = TASKS["tasks"]["bowl_plate"]["task_name"]


def run_libero_set(
    policy: SmolVLAPolicy, preprocessor: ObservationProcessor, postprocessor: ActionProcessor
) -> list[dict[str, int | bool | str]]:
    rows = []
    for state_id in STATE_IDS:
        seed = SEED + state_id
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        environment = open_libero_task("bowl_plate", state_id, CONFIGS, DATA, control_hz=20)
        try:
            result = run_policy_episode(
                environment,
                policy,
                preprocessor,
                postprocessor,
                "bowl_plate",
                state_id,
                CONFIGS,
                partition="development",
                task_prompt=TASK_PROMPT,
                reset_seed=seed,
                control_hz=20,
            )
            rows.append(
                {
                    "state_id": state_id,
                    "seed": seed,
                    "success": result.success,
                    "steps": result.steps,
                    "termination_reason": result.termination_reason,
                    "control_hz": result.control_hz,
                }
            )
        finally:
            environment.close()
    return rows


BASELINE_CALLBACK = CallbackSmolVLAPolicy(
    POLICY, sampler=euler_to_action, input_fn=course_input_fn, num_steps=8
)
BASELINE_EPISODES = run_libero_set(BASELINE_CALLBACK, PREPROCESSOR, POSTPROCESSOR)
print(
    {
        "held_out_loss_before": float(BEFORE_LOSS),
        "baseline_successes": sum(int(row["success"]) for row in BASELINE_EPISODES),
        "episodes": len(BASELINE_EPISODES),
        "state_ids": STATE_IDS,
        "seeds": [row["seed"] for row in BASELINE_EPISODES],
    }
)

This baseline fixes the data windows, noise, time, LIBERO states, and seeds. LIBERO controller rate is 20 Hz. Stored action labels remain at 10 FPS; no label resampling occurs.

In [ ]:
# Expert-only adaptation. Hardware limits: at most 48 GB and two hours.
TRAIN_DS = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "train",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
train_mode_with_frozen_vlm(POLICY)
TRAINABLE = select_trainable_parameters(POLICY)
OPTIMIZER = torch.optim.AdamW(TRAINABLE, lr=1e-5, weight_decay=1e-4, foreach=False)
VLM_BEFORE = [
    parameter.detach().cpu().clone() for parameter in POLICY.model.vlm_with_expert.vlm.parameters()
]
TRAIN_GENERATOR = torch.Generator().manual_seed(SEED)
MAX_SECONDS = 2 * 60 * 60
MAX_UPDATES = 1000
BATCH_SIZE = 1
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    MEMORY_LIMIT_BYTES = min(
        48 * 1024**3, int(torch.cuda.get_device_properties(0).total_memory * 0.85)
    )
else:
    MEMORY_LIMIT_BYTES = 48 * 1024**3
STARTED = time.monotonic()
LOSSES = []
for _step in range(MAX_UPDATES):
    if time.monotonic() - STARTED >= MAX_SECONDS:
        break
    indices = torch.randint(len(TRAIN_DS), (BATCH_SIZE,), generator=TRAIN_GENERATOR).tolist()
    RAW_BATCH = default_collate([TRAIN_DS[index] for index in indices])
    RAW_BATCH["task"] = [TASK_PROMPT] * BATCH_SIZE
    BATCH = PREPROCESSOR(RAW_BATCH)
    BATCH = {
        key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
        for key, value in BATCH.items()
    }
    metrics = optimizer_update(ADAPTER, BATCH, OPTIMIZER, learner_flow_loss)
    LOSSES.append(float(metrics["loss"]))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        if torch.cuda.max_memory_allocated() > MEMORY_LIMIT_BYTES:
            raise MemoryError("The local GPU budget was exceeded; stop and reduce batch size")
assert len(LOSSES) >= 100, (
    "At least 100 real updates are required before this is an adaptation result."
)
ADAPTER.eval()
with torch.no_grad():
    AFTER_LOSS, _ = learner_flow_loss(ADAPTER, VALID_BATCH, FIXED_NOISE, FIXED_TIME)
assert all(
    torch.equal(before, after.detach().cpu())
    for before, after in zip(VLM_BEFORE, POLICY.model.vlm_with_expert.vlm.parameters(), strict=True)
)
ELAPSED_SECONDS = time.monotonic() - STARTED
PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
RESULT_STAMP = time.strftime("%Y%m%d-%H%M%S") + f"-{os.getpid()}"
OUTPUT = WORKSPACE / ".local-data/course-results" / f"smolvla-adaptation-20-{RESULT_STAMP}"
OUTPUT.mkdir(parents=True, exist_ok=False)
POLICY.save_pretrained(OUTPUT)
PREPROCESSOR.save_pretrained(OUTPUT)
POSTPROCESSOR.save_pretrained(OUTPUT)
print(
    {
        "updates": len(LOSSES),
        "seconds": round(ELAPSED_SECONDS, 1),
        "peak_gpu_gb": PEAK_GPU_GB,
        "held_out_loss_before": float(BEFORE_LOSS),
        "held_out_loss_after": float(AFTER_LOSS),
        "frozen_vlm": True,
        "checkpoint": str(OUTPUT),
    }
)

The training loss does not stand in for task success. The next cells reload the model and processors, check output parity, and run the same development states with the same seeds. Both baseline and adapted policies use the same learner input hook and eight-step Euler sampler, so the comparison changes policy weights only.

In [ ]:
# Reload the checkpoint and processors, then compare a fixed policy output.
RELOADED_POLICY = SmolVLAPolicy.from_pretrained(OUTPUT, config=POLICY.config).to(DEVICE).eval()
RELOADED_PRE, RELOADED_POST = load_pretrained_policy_processors(
    str(OUTPUT),
    preprocessor_overrides={
        "tokenizer_processor": {
            "tokenizer_name": str(DATA / "models/smolvlm2-500m-video-instruct")
        },
        "device_processor": {"device": DEVICE},
    },
)
RELOADED_ADAPTER = SmolVLAAdapter(RELOADED_POLICY, input_fn=course_input_fn)
with torch.no_grad():
    RELOADED_LOSS, _ = learner_flow_loss(RELOADED_ADAPTER, VALID_BATCH, FIXED_NOISE, FIXED_TIME)
    SAVED_ACTIONS = ADAPTER.sample_chunk(
        ADAPTER.prepare_inputs(VALID_BATCH), noise=FIXED_NOISE, num_steps=8
    )
    RELOADED_ACTIONS = RELOADED_ADAPTER.sample_chunk(
        RELOADED_ADAPTER.prepare_inputs(VALID_BATCH), noise=FIXED_NOISE, num_steps=8
    )
assert torch.allclose(SAVED_ACTIONS, RELOADED_ACTIONS, atol=1e-5)
assert torch.allclose(AFTER_LOSS, RELOADED_LOSS, atol=1e-6)
ADAPTED_CALLBACK = CallbackSmolVLAPolicy(
    RELOADED_POLICY, sampler=euler_to_action, input_fn=course_input_fn, num_steps=8
)
ADAPTED_EPISODES = run_libero_set(ADAPTED_CALLBACK, RELOADED_PRE, RELOADED_POST)
REPORT = {
    "task": "bowl_plate",
    "task_prompt": TASK_PROMPT,
    "train_split": "train",
    "held_out_split": "validation",
    "dataset_revision": TASKS["dataset_revision"],
    "train_episode_ids": TASKS["tasks"]["bowl_plate"]["train_episode_ids"],
    "validation_episode_ids": TASKS["tasks"]["bowl_plate"]["validation_episode_ids"],
    "data_label_rate_hz": 10,
    "dataset_frame_interval_seconds": FRAME_INTERVAL_SECONDS,
    "action_chunk_size": ACTION_CHUNK_SIZE,
    "simulator_control_hz": 20,
    "seed": SEED,
    "training_config": {
        "learning_rate": 1e-5,
        "weight_decay": 1e-4,
        "batch_size": BATCH_SIZE,
        "max_updates": MAX_UPDATES,
        "max_seconds": MAX_SECONDS,
    },
    "fixed_noise_sha256": hashlib.sha256(FIXED_NOISE.detach().cpu().numpy().tobytes()).hexdigest(),
    "fixed_time_sha256": hashlib.sha256(FIXED_TIME.detach().cpu().numpy().tobytes()).hexdigest(),
    "held_out_indices": VALID_IDS,
    "libero_state_ids": STATE_IDS,
    "libero_seeds": [row["seed"] for row in BASELINE_EPISODES],
    "updates": len(LOSSES),
    "elapsed_seconds": ELAPSED_SECONDS,
    "peak_gpu_gb": PEAK_GPU_GB,
    "held_out_loss_before": float(BEFORE_LOSS),
    "held_out_loss_after": float(RELOADED_LOSS),
    "baseline_episodes": BASELINE_EPISODES,
    "adapted_episodes": ADAPTED_EPISODES,
    "checkpoint": str(OUTPUT),
    "reload_action_parity": True,
}
(OUTPUT / "course_report.json").write_text(json.dumps(REPORT, indent=2, sort_keys=True) + "\n")
print(
    {
        "baseline_successes": sum(int(row["success"]) for row in BASELINE_EPISODES),
        "adapted_successes": sum(int(row["success"]) for row in ADAPTED_EPISODES),
        "episodes_per_policy": len(STATE_IDS),
        "report": str(OUTPUT / "course_report.json"),
    }
)

Review `course_report.json`. It records the train/validation split, label rate, simulator rate, indices, seeds, update count, elapsed time, peak memory, both held-out losses, and every episode outcome. Success rate uses all attempted episodes as its denominator.